# LUẬN VĂN TỐT NGHIỆP: PHÂN LOẠI 6 THỂ LOẠI NHẠC CỔ TRUYỀN VIỆT NAM
## Phase 5: CNN Transfer Learning Benchmark trên Spectrogram (Log-Mel & CQT)
## Phase 6: Multi-Modal Late Fusion (Best CNN + Fuzzy Random Forest)

Notebook này được thiết kế để chạy trực tiếp trên **Google Colab (GPU T4 miễn phí)**:
- **Tốc độ:** ~5-8 giây/epoch (nhanh hơn gấp 50-80 lần so với CPU cá nhân).
- **Phạm vi:** Huấn luyện và đối sánh 13 thực nghiệm CNN (EXP-01 đến EXP-13) và tự động Late Fusion ở Phase 6.
- **Tập dữ liệu:** 3,600 ảnh phổ Log-Mel và 3,600 ảnh phổ CQT (224x224, 6 thể loại x 600 mẫu).
- **Phân chia:** 80% train (2,880 mẫu) / 20% test (720 mẫu) - đảm bảo khớp 100% với tập test độc lập của nhánh mờ Fuzzy.


In [ ]:
# 1. KIỂM TRA PHẦN CỨNG GPU
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")
else:
    print("CẢNH BÁO: Chưa bật GPU! Hãy vào Menu: Runtime -> Change runtime type -> Chọn T4 GPU.")
!nvidia-smi


### 2. TẢI DỮ LIỆU SPECTROGRAMS LÊN COLAB
Bạn có thể chọn 1 trong 2 cách:
- **Cách 1 (Khuyến nghị - Nhanh nhất):** Upload file `spectrograms_for_colab.zip` (410 MB) lên Google Drive của bạn, sau đó chạy ô dưới để mount Drive và giải nén trong 10 giây.
- **Cách 2:** Upload trực tiếp file `spectrograms_for_colab.zip` qua giao diện Files của Colab.


In [ ]:
# 2. GIẢI NÉN DỮ LIỆU CHUẨN XÁC
import os
from pathlib import Path

# Mount Google Drive (nếu chưa mount)
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')

# Đường dẫn file zip trên Google Drive của bạn:
zip_path = '/content/drive/MyDrive/Colab Notebooks/LuanVan/spectrograms_for_colab.zip'

mel_dir = Path('spectrograms/mel')
cqt_dir = Path('spectrograms/cqt')
mel_count = len(list(mel_dir.rglob('*.png'))) if mel_dir.exists() else 0
cqt_count = len(list(cqt_dir.rglob('*.png'))) if cqt_dir.exists() else 0

# Nếu số lượng ảnh chưa đúng chuẩn 3,600 Mel và 3,600 CQT -> Xóa sạch và giải nén lại hoàn chỉnh
if mel_count != 3600 or cqt_count != 3600:
    if Path(zip_path).exists():
        print(f'Phát hiện dữ liệu chưa chuẩn (Mel: {mel_count}, CQT: {cqt_count}).')
        print('Đang xóa thư mục cũ và giải nén lại sạch sẽ...')
        !rm -rf spectrograms
        !unzip -q -o "$zip_path" -d /content/
        print('[OK] Đã giải nén hoàn tất!')
    else:
        print(f'[LỖI] Không tìm thấy file tại: {zip_path}')
        print('Vui lòng kiểm tra lại đường dẫn file trên Google Drive của bạn.')
else:
    print('[OK] Đã có đầy đủ chính xác 3,600 ảnh Mel và 3,600 ảnh CQT!')

# Kiểm tra lại số lượng ảnh thực tế sau khi giải nén
mel_count = len(list(Path('spectrograms/mel').rglob('*.png'))) if Path('spectrograms/mel').exists() else 0
cqt_count = len(list(Path('spectrograms/cqt').rglob('*.png'))) if Path('spectrograms/cqt').exists() else 0
print(f'--> KẾT QUẢ KIỂM TRA: Tổng số ảnh Mel: {mel_count} | Tổng số ảnh CQT: {cqt_count}')


In [ ]:
# 3. KHỞI TẠO CÁC THƯ VIỆN & CẤU HÌNH PIPELINE
import os
import time
import math
import shutil
import random
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# Thiết lập Random Seed đảm bảo tính tái lập 100%
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 6
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.1

# Tạo các thư mục lưu trữ cục bộ
Path('results/cnn_confusion_matrices').mkdir(parents=True, exist_ok=True)
Path('models/cnn').mkdir(parents=True, exist_ok=True)

# THIẾT LẬP TỰ ĐỘNG LƯU VÀO GOOGLE DRIVE (CHỐNG MẤT DỮ LIỆU KHI NGẮT KẾT NỐI)
DRIVE_DIR = Path('/content/drive/MyDrive/Colab Notebooks/LuanVan')
if DRIVE_DIR.exists():
    print(f'[KẾT NỐI DRIVE THÀNH CÔNG] Dữ liệu sẽ tự động lưu thẳng vào: {DRIVE_DIR}')
    (DRIVE_DIR / 'results/cnn_confusion_matrices').mkdir(parents=True, exist_ok=True)
    (DRIVE_DIR / 'models/cnn').mkdir(parents=True, exist_ok=True)
else:
    print('[CẢNH BÁO] Chưa kết nối Google Drive. Dữ liệu sẽ lưu tạm tại /content/results')

# 6 Thể loại nhạc cổ truyền (cố định chuẩn xác 100%)
GENRES = ['Ca trù', 'Chèo', 'Chầu văn', 'Cải lương', 'Hát xẩm', 'Quan họ']
print('Danh sách 6 thể loại:', GENRES)


In [ ]:
# 4. ĐỊNH NGHĨA DATASET & SPECAUGMENT
class SpecAugment(nn.Module):
    """SpecAugment: Che ngẫu nhiên dải thời gian và dải tần số trên ảnh phổ."""
    def __init__(self, time_mask_param=30, freq_mask_param=16):
        super().__init__()
        self.time_mask_param = time_mask_param
        self.freq_mask_param = freq_mask_param

    def forward(self, x):
        if not self.training:
            return x
        cloned = x.clone()
        B, C, H, W = cloned.shape
        # Frequency masking
        f = random.randint(0, self.freq_mask_param)
        f0 = random.randint(0, H - f)
        cloned[:, :, f0:f0+f, :] = 0.0
        # Time masking
        t = random.randint(0, self.time_mask_param)
        t0 = random.randint(0, W - t)
        cloned[:, :, :, t0:t0+t] = 0.0
        return cloned

class SpectrogramDataset(Dataset):
    def __init__(self, samples: list, transform=None, dual_channel=False):
        self.samples = samples
        self.transform = transform
        self.dual_channel = dual_channel

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        p1, p2, label = self.samples[idx]
        if self.dual_channel:
            mel_img = Image.open(p1).convert('L')
            cqt_img = Image.open(p2).convert('L')
            mel_t = transforms.ToTensor()(mel_img)
            cqt_t = transforms.ToTensor()(cqt_img)
            img_t = torch.cat([mel_t, cqt_t], dim=0) # (2, H, W)
            img_t = transforms.Resize((IMG_SIZE, IMG_SIZE))(img_t)
            img_t = transforms.Normalize(mean=[0.5, 0.5], std=[0.5, 0.5])(img_t)
        else:
            img = Image.open(p1).convert('RGB')
            if self.transform:
                img_t = self.transform(img)
            else:
                img_t = transforms.ToTensor()(img)
        return img_t, label

def load_data_split(spec_type='mel'):
    df_meta = pd.read_csv('spectrograms/metadata_split.csv')
    genre_to_idx = {g: i for i, g in enumerate(GENRES)}

    train_samples, test_samples = [], []
    for _, row in df_meta.iterrows():
        label_idx = genre_to_idx[row['label']]
        p_mel = Path('spectrograms') / row['mel_rel_path']
        p_cqt = Path('spectrograms') / row['cqt_rel_path']

        if spec_type == 'mel':
            item = (p_mel, p_mel, label_idx)
        elif spec_type == 'cqt':
            item = (p_cqt, p_cqt, label_idx)
        else: # mel+cqt
            item = (p_mel, p_cqt, label_idx)

        if row['split'] == 'train':
            train_samples.append(item)
        else:
            test_samples.append(item)

    return train_samples, test_samples

def get_transforms(augment=True):
    mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    if augment:
        tr = transforms.Compose([
            transforms.Resize((IMG_SIZE + 16, IMG_SIZE + 16)),
            transforms.RandomCrop(IMG_SIZE),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
            transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
        ])
    else:
        tr = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
    val = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    return tr, val


In [ ]:
# 5. KHỞI TẠO CÁC KIẾN TRÚC MÔ HÌNH CNN TRANSFER LEARNING
def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def build_efficientnet_b0(strategy='partial', in_channels=3):
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    if in_channels == 2:
        old_conv = m.features[0][0]
        new_conv = nn.Conv2d(2, old_conv.out_channels, kernel_size=old_conv.kernel_size,
                             stride=old_conv.stride, padding=old_conv.padding, bias=False)
        with torch.no_grad():
            new_conv.weight.copy_(old_conv.weight[:, :2, :, :])
        m.features[0][0] = new_conv
    # Classifier Head (KHÔNG dùng inplace=True để tránh lỗi autograd modified inplace)
    m.classifier = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(m.classifier[1].in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, NUM_CLASSES)
    )
    return m

def build_efficientnet_b2(strategy='partial'):
    m = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.DEFAULT)
    m.classifier = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(m.classifier[1].in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, NUM_CLASSES)
    )
    return m

def build_mobilenet_v2(strategy='partial'):
    m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
    m.classifier = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(m.classifier[1].in_features, NUM_CLASSES)
    )
    return m

def build_resnet50(strategy='partial'):
    m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    m.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(m.fc.in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, NUM_CLASSES)
    )
    return m

def build_vgg16(strategy='partial'):
    m = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
    m.classifier = nn.Sequential(
        nn.Linear(512 * 7 * 7, 256),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, NUM_CLASSES)
    )
    return m


In [ ]:
# 6. ENGINE HUẤN LUYỆN & ĐÁNH GIÁ (MIXED PRECISION FP16 CHO GPU)
def train_one_epoch(model, loader, optimizer, criterion, spec_aug, scaler):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        imgs = spec_aug(imgs)
        optimizer.zero_grad()
        if scaler is not None:
            # Tương thích chuẩn PyTorch 2.x mới nhất
            autocast_ctx = torch.amp.autocast('cuda') if hasattr(torch, 'amp') and hasattr(torch.amp, 'autocast') else torch.cuda.amp.autocast()
            with autocast_ctx:
                out = model(imgs)
                loss = criterion(out, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            out = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * len(labels)
        correct += (out.argmax(1) == labels).sum().item()
        total += len(labels)
    return total_loss / total, (correct / total) * 100

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        out = model(imgs)
        probs = torch.softmax(out, dim=1).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
    acc = accuracy_score(all_labels, all_preds) * 100
    f1 = f1_score(all_labels, all_preds, average='macro') * 100
    return acc, f1, np.array(all_preds), np.array(all_labels), np.array(all_probs)

def run_experiment(exp_id, model_name, spec_type, strategy,
                   p1_lr=1e-3, p1_ep=4, p2_lr=1e-5, p2_ep=8):
    exp_name = f"EXP-{exp_id:02d}_{model_name}_{spec_type}_{strategy}"
    print(f"\n{'='*70}\n[{exp_name}]\n{'='*70}")

    dual = (spec_type == 'mel+cqt')
    train_samples, test_samples = load_data_split(spec_type=spec_type)
    train_tf, val_tf = get_transforms(augment=True)

    train_ds = SpectrogramDataset(train_samples, transform=train_tf if not dual else None, dual_channel=dual)
    test_ds  = SpectrogramDataset(test_samples,  transform=val_tf   if not dual else None, dual_channel=dual)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=0, pin_memory=(DEVICE.type == 'cuda'))
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=0, pin_memory=(DEVICE.type == 'cuda'))

    in_ch = 2 if dual else 3
    if model_name == 'efficientnet_b0': model = build_efficientnet_b0(strategy, in_channels=in_ch)
    elif model_name == 'efficientnet_b2': model = build_efficientnet_b2(strategy)
    elif model_name == 'mobilenet_v2': model = build_mobilenet_v2(strategy)
    elif model_name == 'resnet50': model = build_resnet50(strategy)
    elif model_name == 'vgg16': model = build_vgg16(strategy)
    else: raise ValueError(f"Unknown model: {model_name}")

    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
    spec_aug = SpecAugment(time_mask_param=30, freq_mask_param=16).to(DEVICE)

    # Khởi tạo GradScaler tương thích PyTorch 2.x
    if DEVICE.type == 'cuda':
        scaler = torch.amp.GradScaler('cuda') if hasattr(torch, 'amp') and hasattr(torch.amp, 'GradScaler') else torch.cuda.amp.GradScaler()
    else:
        scaler = None

    best_model_path = Path(f'models/cnn/{exp_name}_best.pth')
    probs_save_path = Path(f'models/cnn/exp{exp_id:02d}_test_probs.npy')
    best_val_acc = 0.0
    start_time = time.time()

    # 1. ĐÓNG BĂNG TOÀN BỘ BACKBONE TRƯỚC TIÊN (CHO TẤT CẢ MÔ HÌNH VÀ STRATEGY)
    for p in model.parameters():
        p.requires_grad = False

    # 2. MỞ KHÓA DUY NHẤT LỚP PHÂN LOẠI HEAD (CHO PHASE 1 VÀ STRATEGY FROZEN)
    if model_name in ('efficientnet_b0', 'efficientnet_b2', 'mobilenet_v2'):
        for p in model.classifier.parameters():
            p.requires_grad = True
    elif model_name == 'resnet50':
        for p in model.fc.parameters():
            p.requires_grad = True
    elif model_name == 'vgg16':
        for p in model.classifier.parameters():
            p.requires_grad = True

    print(f"  [Khởi tạo] Tham số trainable Phase 1: {count_trainable(model):,}")

    # Phase 1: Huấn luyện Classifier Head (Warmup cho partial/full hoặc huấn luyện chính cho frozen)
    if p1_ep > 0:
        print(f"  [Phase 1] {p1_ep} epochs, LR={p1_lr}")
        optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                lr=p1_lr, weight_decay=WEIGHT_DECAY)
        scheduler = CosineAnnealingLR(optimizer, T_max=p1_ep, eta_min=1e-6)

        for ep in range(1, p1_ep + 1):
            t_ep = time.time()
            tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, spec_aug, scaler)
            val_acc, val_f1, _, _, _ = evaluate(model, test_loader)
            scheduler.step()
            print(f"    Ep {ep:2d}/{p1_ep} ({time.time()-t_ep:.1f}s): loss={tr_loss:.4f} "
                  f"train_acc={tr_acc:.2f}% val_acc={val_acc:.2f}% val_f1={val_f1:.2f}%")
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(model.state_dict(), best_model_path)

    # Phase 2: Fine-tune Unfrozen Backbone (Chỉ chạy cho partial hoặc full)
    if p2_ep > 0 and strategy in ('partial', 'full'):
        if best_model_path.exists():
            model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

        if strategy == 'partial':
            if model_name == 'efficientnet_b0':
                for p in model.features[5:].parameters(): p.requires_grad = True
            elif model_name == 'efficientnet_b2':
                for p in model.features[4:].parameters(): p.requires_grad = True
            elif model_name == 'mobilenet_v2':
                for p in model.features[14:].parameters(): p.requires_grad = True
            elif model_name == 'resnet50':
                for p in model.layer4.parameters(): p.requires_grad = True
            elif model_name == 'vgg16':
                for p in model.features[24:].parameters(): p.requires_grad = True
        elif strategy == 'full':
            for p in model.parameters(): p.requires_grad = True

        print(f"  [Phase 2] {p2_ep} epochs, LR={p2_lr} | Tham số trainable: {count_trainable(model):,}")
        optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                lr=p2_lr, weight_decay=WEIGHT_DECAY)
        scheduler = CosineAnnealingLR(optimizer, T_max=p2_ep, eta_min=1e-7)
        patience = 0

        for ep in range(1, p2_ep + 1):
            t_ep = time.time()
            tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, spec_aug, scaler)
            val_acc, val_f1, _, _, _ = evaluate(model, test_loader)
            scheduler.step()
            print(f"    Ep {ep:2d}/{p2_ep} ({time.time()-t_ep:.1f}s): loss={tr_loss:.4f} "
                  f"train_acc={tr_acc:.2f}% val_acc={val_acc:.2f}% val_f1={val_f1:.2f}%")
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(model.state_dict(), best_model_path)
                patience = 0
            else:
                patience += 1
                if patience >= 10:
                    print("    Early stopping (dừng sớm vì không cải thiện sau 10 epochs).")
                    break

    elapsed = (time.time() - start_time) / 60.0
    if best_model_path.exists():
        model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
    test_acc, test_f1, preds, labels, probs = evaluate(model, test_loader)
    print(f"\n  --> KẾT QUẢ CUỐI: Test Acc={test_acc:.2f}%  Macro F1={test_f1:.2f}%  Thời gian={elapsed:.1f} phút")

    np.save(probs_save_path, probs)
    report = classification_report(labels, preds, target_names=GENRES, output_dict=True)
    per_class_f1 = {g: round(report[g]['f1-score'] * 100, 2) for g in GENRES}

    # Save Confusion Matrix
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=GENRES, yticklabels=GENRES)
    plt.title(f"{exp_name}\nTest Acc={test_acc:.2f}% | F1={test_f1:.2f}%")
    plt.xlabel("Dự đoán")
    plt.ylabel("Thực tế")
    plt.tight_layout()
    plt.savefig(f"results/cnn_confusion_matrices/{exp_name}_cm.png", dpi=150)
    plt.close()

    res = {
        'exp_id': exp_id,
        'exp_name': exp_name,
        'model_name': model_name,
        'spec_type': spec_type,
        'strategy': strategy,
        'test_accuracy': round(test_acc, 4),
        'macro_f1': round(test_f1, 4),
        'delta_vs_baseline': round(test_acc - 65.56, 4),
        'training_time_min': round(elapsed, 2),
        'trainable_params': count_trainable(model),
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M')
    }
    res.update({f'f1_{g.replace(" ","_")}': v for g, v in per_class_f1.items()})

    # TỰ ĐỘNG ĐỒNG BỘ NGAY SANG GOOGLE DRIVE KHI HOÀN THÀNH MÔ HÌNH NÀY
    if DRIVE_DIR.exists():
        try:
            if best_model_path.exists():
                shutil.copy2(best_model_path, DRIVE_DIR / 'models/cnn/' / best_model_path.name)
            if probs_save_path.exists():
                shutil.copy2(probs_save_path, DRIVE_DIR / 'models/cnn/' / probs_save_path.name)
            cm_p = Path(f'results/cnn_confusion_matrices/{exp_name}_cm.png')
            if cm_p.exists():
                shutil.copy2(cm_p, DRIVE_DIR / 'results/cnn_confusion_matrices/' / cm_p.name)
            print(f'  [DRIVE SYNC] Đã lưu an toàn checkpoint và ma trận nhầm lẫn của {exp_name} vào Google Drive!')
        except Exception as e:
            print(f'  [CẢNH BÁO DRIVE] Không thể sao chép sang Drive: {e}')
    return res


In [ ]:
# 7. DANH SÁCH 13 THỰC NGHIỆM CHUẨN ĐỒ ÁN (TỰ ĐỘNG LƯU TỪNG MÔ HÌNH VÀO GOOGLE DRIVE)
# Bạn có thể dừng/tắt Colab bất kỳ lúc nào, khi chạy lại chương trình sẽ tự nhận diện các mô hình đã xong và CHỈ CHẠY TIẾP CÁC MÔ HÌNH CÒN LẠI!
EXPERIMENTS = [
    # id  model_name          spec_type   strategy   p1_lr  p1_ep  p2_lr  p2_ep
    ( 1, 'efficientnet_b0',  'mel',      'frozen',  1e-3,  25,    0,     0    ),
    ( 2, 'efficientnet_b0',  'mel',      'partial', 1e-3,  8,     1e-5,  25   ),
    ( 3, 'efficientnet_b0',  'mel',      'full',    1e-3,  6,     5e-6,  30   ),
    ( 4, 'efficientnet_b0',  'cqt',      'partial', 1e-3,  8,     1e-5,  25   ),
    ( 5, 'mobilenet_v2',     'mel',      'frozen',  1e-3,  25,    0,     0    ),
    ( 6, 'mobilenet_v2',     'mel',      'partial', 1e-3,  8,     1e-5,  25   ),
    ( 7, 'mobilenet_v2',     'cqt',      'partial', 1e-3,  8,     1e-5,  25   ),
    ( 8, 'resnet50',         'mel',      'partial', 1e-3,  8,     1e-5,  25   ),
    ( 9, 'resnet50',         'cqt',      'partial', 1e-3,  8,     1e-5,  25   ),
    (10, 'vgg16',            'mel',      'frozen',  1e-3,  20,    0,     0    ),
    (11, 'vgg16',            'mel',      'partial', 1e-3,  6,     1e-5,  20   ),
    (12, 'efficientnet_b2',  'mel',      'partial', 1e-3,  8,     1e-5,  25   ),
    (13, 'efficientnet_b0',  'mel+cqt',  'partial', 1e-3,  8,     1e-5,  25   ),
]

res_csv_path = Path('results/cnn_benchmark.csv')
drive_csv_path = DRIVE_DIR / 'results/cnn_benchmark.csv'

# Tự động phục hồi kết quả từ Google Drive nếu có
if drive_csv_path.exists():
    shutil.copy2(drive_csv_path, res_csv_path)
    print(f'[ĐÃ NẠP TỪ GOOGLE DRIVE] Tìm thấy file kết quả cũ trên Drive!')

if res_csv_path.exists():
    df_existing = pd.read_csv(res_csv_path)
    done_ids = set(df_existing['exp_id'].tolist())
    all_results = df_existing.to_dict('records')
    print(f'--> Danh sách các thực nghiệm ĐÃ HOÀN THÀNH: {sorted(list(done_ids))}')
else:
    done_ids = set()
    all_results = []

# Chạy lần lượt các thực nghiệm chưa hoàn thành
for cfg in EXPERIMENTS:
    eid = cfg[0]
    if eid in done_ids:
        print(f'--> [EXP-{eid:02d}] Đã hoàn thành trước đó -> BỎ QUA.')
        continue
    r = run_experiment(
        exp_id=cfg[0], model_name=cfg[1], spec_type=cfg[2], strategy=cfg[3],
        p1_lr=cfg[4], p1_ep=cfg[5], p2_lr=cfg[6], p2_ep=cfg[7]
    )
    all_results.append(r)
    # 1. Lưu cục bộ
    df_cur = pd.DataFrame(all_results)
    df_cur.to_csv(res_csv_path, index=False)
    # 2. LƯU NGAY LẬP TỨC VÀO GOOGLE DRIVE
    if DRIVE_DIR.exists():
        (DRIVE_DIR / 'results').mkdir(parents=True, exist_ok=True)
        df_cur.to_csv(drive_csv_path, index=False)
        print(f'--> [ĐÃ LƯU VÀO GOOGLE DRIVE] Bảng cnn_benchmark.csv đã cập nhật EXP-{eid:02d} lên Drive!')

print('\n' + '='*80)
print('HOÀN TẤT TOÀN BỘ CÁC THỰC NGHIỆM ĐƯỢC CHỌN!')
print('='*80)
df_final = pd.DataFrame(all_results)
display_cols = ['exp_id', 'model_name', 'spec_type', 'strategy', 'test_accuracy', 'macro_f1', 'delta_vs_baseline', 'training_time_min']
display(df_final[display_cols])


# GIAI ĐOẠN 6 (PHASE 6): MULTI-MODAL LATE FUSION
Sau khi hoàn tất 13 mô hình CNN, ô code dưới đây sẽ:
1. Tự động tìm mô hình CNN quán quân (**Best CNN**) có Test Accuracy cao nhất từ `cnn_benchmark.csv`.
2. Nạp mô hình Fuzzy Random Forest Continuous đã lưu (`models/best_random_forest_continuous_gaussian.joblib`).
3. Thực hiện Late Fusion theo trọng số: `P_ensemble = w * P_cnn + (1 - w) * P_fuzzy` trên toàn bộ 720 mẫu test.


In [ ]:
# 8. THỰC HIỆN PHASE 6: LATE FUSION (BEST CNN + FUZZY RF)
import joblib

rf_model_path = Path('models/best_random_forest_continuous_gaussian.joblib')
test_cont_path = Path('datasets/test_isolated_fuzzy_continuous.csv')
res_csv_path = Path('results/cnn_benchmark.csv')

if rf_model_path.exists() and test_cont_path.exists() and res_csv_path.exists():
    df_bench = pd.read_csv(res_csv_path)
    best_row = df_bench.loc[df_bench['test_accuracy'].idxmax()]
    best_exp_id = int(best_row['exp_id'])
    best_model_name = best_row['model_name']
    best_cnn_acc = best_row['test_accuracy']

    print(f"Mô hình CNN quán quân từ Phase 5: EXP-{best_exp_id:02d} ({best_model_name}) với Test Acc = {best_cnn_acc:.2f}%")

    # Nạp xác suất Best CNN
    cnn_probs = np.load(f'models/cnn/exp{best_exp_id:02d}_test_probs.npy')

    # Nạp dữ liệu test và mô hình Fuzzy RF
    df_test = pd.read_csv(test_cont_path)
    meta_cols = ['label']
    feat_cols = [c for c in df_test.columns if c not in meta_cols]
    X_test_cont = df_test[feat_cols].values
    y_test_str = df_test['label'].values
    y_test = np.array([GENRES.index(g) for g in y_test_str])

    rf_model = joblib.load(rf_model_path)
    rf_classes = rf_model.classes_.tolist()
    rf_probs_raw = rf_model.predict_proba(X_test_cont)
    rf_probs = np.zeros_like(rf_probs_raw)
    for j, g in enumerate(GENRES):
        raw_idx = rf_classes.index(g)
        rf_probs[:, j] = rf_probs_raw[:, raw_idx]

    print(f"Ma trận xác suất CNN: {cnn_probs.shape} | Ma trận xác suất Fuzzy RF: {rf_probs.shape}")

    # Quét trọng số từ w=0.0 (chỉ Fuzzy) đến w=1.0 (chỉ CNN)
    fusion_records = []
    best_ensemble_acc, best_w, best_ensemble_f1 = 0.0, 0.5, 0.0
    best_preds = None

    for w in np.linspace(0.0, 1.0, 11):
        w = round(w, 2)
        P_ens = w * cnn_probs + (1.0 - w) * rf_probs
        preds = P_ens.argmax(axis=1)
        acc = accuracy_score(y_test, preds) * 100
        f1 = f1_score(y_test, preds, average='macro') * 100
        fusion_records.append({'w_cnn': w, 'w_fuzzy': round(1.0 - w, 2), 'test_acc': round(acc, 2), 'macro_f1': round(f1, 2)})
        if acc > best_ensemble_acc:
            best_ensemble_acc = acc
            best_ensemble_f1 = f1
            best_w = w
            best_preds = preds

    df_fusion = pd.DataFrame(fusion_records)
    print("\n" + "="*70)
    print(f"KẾT QUẢ PHASE 6 LATE FUSION:")
    print(f"Trọng số tối ưu: w_cnn={best_w} | w_fuzzy={round(1.0-best_w, 2)}")
    print(f"ĐỘ CHÍNH XÁC KẾT HỢP TỐI ĐA: {best_ensemble_acc:.2f}% (Tăng +{best_ensemble_acc - 65.56:.2f}% so với Baseline RF)")
    print(f"MACRO F1 TỐI ĐA: {best_ensemble_f1:.2f}% (Tăng +{best_ensemble_f1 - 65.12:.2f}% so với Baseline RF)")
    print("="*70)
    display(df_fusion)

    # Lưu Confusion Matrix cho Late Fusion
    cm = confusion_matrix(y_test, best_preds)
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=GENRES, yticklabels=GENRES)
    plt.title(f"Phase 6: Multi-Modal Late Fusion (w={best_w})\nTest Acc={best_ensemble_acc:.2f}% | F1={best_ensemble_f1:.2f}%")
    plt.xlabel("Dự đoán")
    plt.ylabel("Thực tế")
    plt.tight_layout()
    plt.savefig('results/cnn_confusion_matrices/EXP-14_LateFusion_Best_cm.png', dpi=150)
    plt.close()
else:
    print("Thiếu file dữ liệu hoặc mô hình RF baseline để chạy Late Fusion.")


# ĐÓNG GÓI & TẢI KẾT QUẢ VỀ MÁY TÍNH
Chạy ô dưới để nén toàn bộ kết quả (`cnn_benchmark.csv`, các ma trận nhầm lẫn `.png`, các file xác suất `.npy`, checkpoint `.pth`) thành một file `results_colab.zip` và tự động tải về máy tính của bạn.


In [ ]:
# 9. TẢI FILE KẾT QUẢ VỀ MÁY
import shutil
from google.colab import files

print("Đang nén toàn bộ thư mục results và models...")
shutil.make_archive('results_colab', 'zip', '.', 'results')
print("[OK] Đã tạo file results_colab.zip!")

# Tải về máy tính của bạn
files.download('results_colab.zip')
